# PCA and Spatial Correlation Inference Pipeline

This notebook computes PCA using the nine `scaled_deltaW` period columns from `data/data.csv`, saves the PCA outputs in `pca_results/`, and prepares posterior inference for `normalized_PC1` through `normalized_PC9` using both `modelE` and `modelEAS`.

## Cell 1: Import packages for PCA

This cell imports the packages used to read the input data, run PCA, and save the PCA outputs.

In [25]:
from pathlib import Path

import pandas as pd
from sklearn.decomposition import PCA

## Cell 2: Load the nine scaled_deltaW period columns

This cell defines the input file, creates the output folders, selects the nine `scaled_deltaW` columns, assigns the period labels used in the exported loadings table, and shows the first few rows of the PCA input matrix.

In [26]:
data_path = Path('data/data.csv')
pca_output_dir = Path('pca_results')
inference_output_dir = Path('inference_results')
pca_output_dir.mkdir(parents=True, exist_ok=True)
inference_output_dir.mkdir(parents=True, exist_ok=True)

period_columns = [
    'scaled_deltaWT001',
    'scaled_deltaWT003',
    'scaled_deltaWT006',
    'scaled_deltaWT010',
    'scaled_deltaWT030',
    'scaled_deltaWT060',
    'scaled_deltaWT100',
    'scaled_deltaWT300',
    'scaled_deltaWT600',
]
period_labels = ['0.01', '0.03', '0.06', '0.1', '0.3', '0.6', '1', '3', '6']

df = pd.read_csv(data_path)
X = df[period_columns]

X.head()

,scaled_deltaWT001,scaled_deltaWT003,scaled_deltaWT006,scaled_deltaWT010,scaled_deltaWT030,scaled_deltaWT060,scaled_deltaWT100,scaled_deltaWT300,scaled_deltaWT600
0,-0.762490,-0.837575,-1.058218,-1.281882,-0.535206,0.388256,0.596367,0.656036,-0.044771
1,0.586623,0.603085,0.691974,0.802130,0.670707,0.754101,0.404635,-0.373843,1.257506
2,-0.320868,-0.392246,-0.428254,-0.116149,0.702226,-0.041146,-0.336795,-0.260693,1.339455
3,-0.197856,-0.282714,-0.510113,-0.488122,0.363372,0.167172,0.781085,1.606929,0.811405
4,-0.328049,-0.421697,-0.669818,-0.600869,0.533561,0.121033,0.350093,1.066029,0.168708


## Cell 3: Fit PCA and save the required outputs

This cell fits PCA to the nine period columns, saves the PCA loadings and explained variance ratio, computes normalized PC scores, appends the normalized PC scores to the original data table, writes the required PCA outputs to `pca_results/`, and displays the main PCA result tables.

In [27]:
pca = PCA(n_components=len(period_columns))
pc_scores_array = pca.fit_transform(X)

pc_columns = [f'PC{i}' for i in range(1, len(period_columns) + 1)]
normalized_pc_columns = [f'normalized_PC{i}' for i in range(1, len(period_columns) + 1)]

loadings = pd.DataFrame(
    pca.components_.T,
    index=period_labels,
    columns=pc_columns,
)

variance_ratio = pd.DataFrame({
    'principal_component': pc_columns,
    'explained_variance_ratio': pca.explained_variance_ratio_,
})

pc_scores = pd.DataFrame(pc_scores_array, columns=pc_columns)
normalized_pc_scores = (pc_scores - pc_scores.mean()) / pc_scores.std(ddof=0)
normalized_pc_scores.columns = normalized_pc_columns
data_with_normalized_pc_scores = pd.concat([df.reset_index(drop=True), normalized_pc_scores], axis=1)

loadings_path = pca_output_dir / 'pca_loadings.csv'
variance_ratio_path = pca_output_dir / 'pca_variance_ratio.csv'
data_with_normalized_pc_scores_path = pca_output_dir / 'data_with_normalized_pc_scores.csv'

loadings.to_csv(loadings_path, index_label='period')
variance_ratio.to_csv(variance_ratio_path, index=False)
data_with_normalized_pc_scores.to_csv(data_with_normalized_pc_scores_path, index=False)

print(f'Loadings saved to: {loadings_path}')
print(f'Variance ratio saved to: {variance_ratio_path}')
print(f'Data with normalized PC scores saved to: {data_with_normalized_pc_scores_path}')

loadings, variance_ratio

Loadings saved to: pca_results\pca_loadings.csv
Variance ratio saved to: pca_results\pca_variance_ratio.csv
Data with normalized PC scores saved to: pca_results\data_with_normalized_pc_scores.csv


(           PC1       PC2       PC3       PC4       PC5       PC6       PC7  \
 0.01  0.406814 -0.167166  0.056798 -0.022901 -0.014842  0.012408 -0.337254   
 0.03  0.401435 -0.181233  0.093874 -0.064709 -0.041413  0.024238 -0.371850   
 0.06  0.380465 -0.254544  0.233465 -0.211383 -0.122371  0.053283 -0.261209   
 0.1   0.362292 -0.292247  0.281634 -0.166508 -0.067458 -0.007133  0.807235   
 0.3   0.354349 -0.100004 -0.221932  0.634799  0.480991 -0.383179  0.078199   
 0.6   0.332031  0.168834 -0.514704  0.183094 -0.163859  0.716381  0.133857   
 1     0.287034  0.356295 -0.479209 -0.372188 -0.329776 -0.555954  0.062699   
 3     0.222269  0.562183  0.208538 -0.369543  0.655792  0.155543  0.012540   
 6     0.172462  0.554534  0.516560  0.459675 -0.426053 -0.055332  0.002678   
 
            PC8       PC9  
 0.01 -0.489826 -0.669898  
 0.03 -0.336923  0.734107  
 0.06  0.770610 -0.106677  
 0.1  -0.140228  0.020010  
 0.3   0.165719  0.021330  
 0.6   0.069014  0.008653  
 1     0.026

## Cell 4: Import packages for posterior inference

This cell imports the packages used to run Bayesian inference with `modelE` and `modelEAS` across all events.

In [28]:
import numpy as np

import jax
import jax.numpy as jnp
import numpyro
from numpyro.infer import MCMC, NUTS

from models_gpu import modelE, modelEAS, precompute_mats

## Cell 5: Precompute event-level matrices once

This cell groups the normalized PCA data by `eqid`, precomputes the event-level distance matrices once, and stores them for reuse across all principal components and both models.

In [29]:
grouped = data_with_normalized_pc_scores.groupby('eqid')
eqids = list(grouped.size().index)
eqids_array = np.asarray(eqids)

event_mats = {}
for eqid in eqids:
    group = grouped.get_group(eqid).copy()
    X_event = group[['epi_dist', 'epi_azimuth', 'vs30']].to_numpy()
    event_mats[eqid] = precompute_mats(X_event)

print(f'Number of events prepared for inference: {len(eqids)}')

Number of events prepared for inference: 41


## Cell 6: Build reusable inference inputs for all normalized PCs

This cell builds the event-wise response vectors for `normalized_PC1` through `normalized_PC9` and stores the input dictionaries needed by `modelE` and `modelEAS`.

In [30]:
inference_inputs = {}

for target_pc in normalized_pc_columns:
    ztot = []
    for eqid in eqids:
        group = grouped.get_group(eqid)
        ztot.append(jnp.asarray(group[target_pc].to_numpy(dtype=np.float64)))

    distsE = [event_mats[eqid]['distE'] for eqid in eqids]
    distsAdeg = [event_mats[eqid]['angdeg'] for eqid in eqids]
    distsS = [event_mats[eqid]['soil'] for eqid in eqids]

    inference_inputs[target_pc] = {
        'modelE': {
            'distsE': distsE,
            'eqids': eqids_array,
            'z': ztot,
        },
        'modelEAS': {
            'distsE': distsE,
            'distsAdeg': distsAdeg,
            'distsS': distsS,
            'eqids': eqids_array,
            'z': ztot,
        },
    }

list(inference_inputs.keys())

['normalized_PC1',
 'normalized_PC2',
 'normalized_PC3',
 'normalized_PC4',
 'normalized_PC5',
 'normalized_PC6',
 'normalized_PC7',
 'normalized_PC8',
 'normalized_PC9']

## Cell 7: Configure MCMC settings

This cell defines the shared MCMC settings used for all principal components and both spatial correlation models.

In [31]:
platform = 'gpu' if jax.default_backend() == 'gpu' else 'cpu'
numpyro.set_platform(platform)

num_chains = 1
num_warmup = 500
num_samples = 500
base_seed = 31

model_specs = {
    'modelE': {
        'model': modelE,
        'parameters': ['LE', 'gammaE'],
    },
    'modelEAS': {
        'model': modelEAS,
        'parameters': ['LE', 'gammaE', 'LA', 'LS', 'w'],
    },
}

print(f'Platform: {platform}')
print(f'Number of target PCs: {len(normalized_pc_columns)}')
print(f'Models: {list(model_specs.keys())}')

Platform: cpu
Number of target PCs: 9
Models: ['modelE', 'modelEAS']


## Cell 8: Run posterior inference for all normalized PCs and both models

This cell loops over `normalized_PC1` through `normalized_PC9` and runs both `modelE` and `modelEAS`. Each posterior table is saved in `inference_results/` using a separate CSV file.

In [ ]:
for pc_index, target_pc in enumerate(normalized_pc_columns):
    for model_index, (model_name, spec) in enumerate(model_specs.items()):
        rng_key = jax.random.PRNGKey(base_seed + 100 * pc_index + model_index)

        mcmc = MCMC(
            NUTS(spec['model']),
            num_warmup=num_warmup,
            num_samples=num_samples,
            num_chains=num_chains,
            chain_method='vectorized',
        )

        mcmc.run(rng_key, **inference_inputs[target_pc][model_name])
        samples = mcmc.get_samples()

        posterior_df = pd.DataFrame({
            parameter: np.asarray(samples[parameter])
            for parameter in spec['parameters']
        })

        posterior_path = inference_output_dir / f'{model_name}_{target_pc}_posterior.csv'
        posterior_df.to_csv(posterior_path, index=False)
        print(f'Saved {model_name} posterior for {target_pc} to: {posterior_path}')